In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
DRIVE_BASE = "/content/drive/MyDrive"
print("Drive mounted:", DRIVE_BASE)


In [ ]:
!pip -q install -U datasets transformers accelerate scikit-learn scipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 49.0 MB/s eta 0:00:00


In [ ]:
import os, re, glob, math, json, random, hashlib
import numpy as np
import torch

from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding, EarlyStoppingCallback
)
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from scipy.special import softmax

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("seed:", SEED)


seed: 42


In [ ]:
SUBTASK = "A"
MODEL_NAME_OR_PATH = "microsoft/codebert-base"
MAX_LEN = 384
DROPOUT = 0.20
LABEL_SMOOTH = 0.08
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0
EPOCHS = 2
BSZ_TRAIN = 32
BSZ_EVAL = 64
GRAD_ACC = 1
FREEZE_EMBEDDINGS = True
FREEZE_FIRST_N_LAYERS = 6
LR_EARLY = 5e-7
LR_LATE  = 8e-6
LR_HEAD  = 1.5e-5
RUN_DIR = f"{DRIVE_BASE}/semeval_codebert_full500k_plus_bigaug_v3"
os.makedirs(RUN_DIR, exist_ok=True)

SAVE_STEPS = 1200
EVAL_STEPS = 600
SAVE_TOTAL_LIMIT = 3

OOD_DEV_TOTAL = 8000
AUG_TARGET_TRAIN = 500_000

print("RUN_DIR:", RUN_DIR)


RUN_DIR: /content/drive/MyDrive/semeval_codebert_full500k_plus_bigaug_v3


In [ ]:
AUG_JSONL_FILES = [
    f"{DRIVE_BASE}/machine_translate_unseen_2400.jsonl",
    f"{DRIVE_BASE}/machine_domain_shift_1500.jsonl",
    f"{DRIVE_BASE}/human_simple_aug_3000.jsonl",
    f"{DRIVE_BASE}/human_domain_aug_2000.jsonl",
]

LLM_AS_HUMAN_FILES = [
    # f"{DRIVE_BASE}/human_translate_student_750.jsonl",
    # f"{DRIVE_BASE}/human_translate_student_750_v2.jsonl",
]
LLM_AS_HUMAN_CAP_TOTAL = 200

existing, missing = [], []
for p in AUG_JSONL_FILES + LLM_AS_HUMAN_FILES:
    if os.path.exists(p):
        existing.append(p)
    else:
        missing.append(p)

print("Existing JSONL:", len(existing))
for p in existing:
    print("  ✅", os.path.basename(p))

if missing:
    print("Missing (ignored):", len(missing))
    for p in missing:
        print("  ⚠️", os.path.basename(p))


Existing JSONL: 4
  ✅ machine_translate_unseen_2400.jsonl
  ✅ machine_domain_shift_1500.jsonl
  ✅ human_simple_aug_3000.jsonl
  ✅ human_domain_aug_2000.jsonl


In [ ]:
base = load_dataset("DaniilOr/SemEval-2026-Task13", SUBTASK)

train_base = base["train"]
val_official = base["validation"]
test_official = base["test"]

print("train_base:", len(train_base), "| cols:", train_base.column_names)
print("val_official:", len(val_official), "| cols:", val_official.column_names)
print("test_official:", len(test_official), "| cols:", test_official.column_names)


train_base: 500000 | cols: ['code', 'generator', 'label', 'language']
val_official: 100000 | cols: ['code', 'generator', 'label', 'language']
test_official: 1000 | cols: ['code', 'generator', 'label', 'language']


In [ ]:
def norm_code(s: str) -> str:
    s = (s or "").replace("\r\n", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def md5_code(s: str) -> str:
    return hashlib.md5(norm_code(s).encode("utf-8", errors="ignore")).hexdigest()

def latest_checkpoint(path: str):
    cks = sorted(
        glob.glob(os.path.join(path, "checkpoint-*")),
        key=lambda p: int(p.split("-")[-1]) if p.split("-")[-1].isdigit() else -1
    )
    return cks[-1] if cks else None

def best_or_latest_checkpoint(run_dir: str):
    ts = os.path.join(run_dir, "trainer_state.json")
    if os.path.exists(ts):
        with open(ts, "r", encoding="utf-8") as f:
            st = json.load(f)
        best = st.get("best_model_checkpoint", None)
        if best and os.path.exists(best):
            return best
    return latest_checkpoint(run_dir)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1_macro": f1_score(labels, preds, average="macro")}


In [ ]:
from datasets import load_dataset as hf_load_dataset

def load_jsonl_min(path: str):
    ds_aug = hf_load_dataset("json", data_files=path, split="train")
    keep = [c for c in ["code", "label", "language", "target_domain", "source_language", "generator"] if c in ds_aug.column_names]
    ds_aug = ds_aug.remove_columns([c for c in ds_aug.column_names if c not in keep])

    ds_aug = ds_aug.filter(lambda x: x.get("code") is not None and len(str(x["code"]).strip()) > 0)
    ds_aug = ds_aug.map(lambda x: {"code": str(x["code"]), "label": int(x["label"])})
    return ds_aug

aug_parts = []
for p in existing:
    d = load_jsonl_min(p)
    aug_parts.append(d)
    print("✅ Loaded", os.path.basename(p), "rows:", len(d))
llm_parts = []
for p in LLM_AS_HUMAN_FILES:
    if os.path.exists(p):
        d = load_jsonl_min(p)
        llm_parts.append(d)
        print("⚠️ Loaded LLM-as-human", os.path.basename(p), "rows:", len(d))

if llm_parts:
    llm_all = concatenate_datasets(llm_parts)
    if len(llm_all) > LLM_AS_HUMAN_CAP_TOTAL:
        llm_all = llm_all.shuffle(seed=SEED).select(range(LLM_AS_HUMAN_CAP_TOTAL))
    aug_parts.append(llm_all)
    print("⚠️ Using LLM-as-human capped:", len(llm_all))

if not aug_parts:
    raise ValueError("No augmentation files loaded. Check paths in AUG_JSONL_FILES.")

aug_all = concatenate_datasets(aug_parts).shuffle(seed=SEED)
print("Aug total loaded:", len(aug_all))


✅ Loaded machine_translate_unseen_2400.jsonl rows: 2400
✅ Loaded machine_domain_shift_1500.jsonl rows: 1500
✅ Loaded human_simple_aug_3000.jsonl rows: 3000
✅ Loaded human_domain_aug_2000.jsonl rows: 2000
Aug total loaded: 8900


In [ ]:

dev_n = min(OOD_DEV_TOTAL, max(1000, len(aug_all)//5))
ood_dev = aug_all.select(range(dev_n)).shuffle(seed=SEED)
aug_train_rest = aug_all.select(range(dev_n, len(aug_all))).shuffle(seed=SEED)
print("OOD dev:", len(ood_dev))
print("Aug train rest:", len(aug_train_rest))


OOD dev: 1780
Aug train rest: 7120


In [ ]:
def upsample_to(ds, target_n: int, seed: int = 42):
    if target_n is None or target_n <= 0:
        return ds
    if len(ds) == 0:
        return ds
    if len(ds) >= target_n:
        return ds.shuffle(seed=seed).select(range(target_n))
    reps = (target_n + len(ds) - 1) // len(ds)
    parts = []
    for r in range(reps):
        parts.append(ds.shuffle(seed=seed + r))
    big = concatenate_datasets(parts)
    return big.select(range(target_n))

aug_big = upsample_to(aug_train_rest, AUG_TARGET_TRAIN, seed=SEED+123)
print("Aug after upsample:", len(aug_big))


Aug after upsample: 500000


In [ ]:
val_hashes = set(md5_code(c) for c in val_official["code"])
print("val hashes:", len(val_hashes))

def not_in_val(ex):
    return md5_code(ex["code"]) not in val_hashes
aug_big = aug_big.filter(not_in_val)
ood_dev = ood_dev.filter(not_in_val)

print("Aug after de-dupe vs val:", len(aug_big))
print("OOD dev after de-dupe vs val:", len(ood_dev))


val hashes: 99999
Aug after de-dupe vs val: 500000
OOD dev after de-dupe vs val: 1780


In [ ]:

train_id_full = train_base
train_full = concatenate_datasets([train_id_full, aug_big]).shuffle(seed=SEED)
print("Train ID full:", len(train_id_full))
print("Train aug big:", len(aug_big))
print("Train FULL:", len(train_full))
print("OOD dev:", len(ood_dev))


Train ID full: 500000
Train aug big: 500000
Train FULL: 1000000
OOD dev: 1780


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH, use_fast=True)

def preprocess(batch):
    return tokenizer(batch["code"], truncation=True, max_length=MAX_LEN, padding=False)
train_tok = train_full.map(preprocess, batched=True)
dev_tok   = ood_dev.map(preprocess, batched=True)
keep_cols = ["input_ids", "attention_mask", "label"]
train_tok = train_tok.remove_columns([c for c in train_tok.column_names if c not in keep_cols])
dev_tok   = dev_tok.remove_columns([c for c in dev_tok.column_names if c not in keep_cols])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenized train/dev:", len(train_tok), len(dev_tok))


Map:   0%|          | 0/1780 [00:00<?, ? examples/s]

Tokenized train/dev: 1000000 1780


In [ ]:

config = AutoConfig.from_pretrained(MODEL_NAME_OR_PATH, num_labels=2)
if hasattr(config, "hidden_dropout_prob"):
    config.hidden_dropout_prob = DROPOUT
if hasattr(config, "attention_probs_dropout_prob"):
    config.attention_probs_dropout_prob = DROPOUT
if hasattr(config, "dropout"):
    config.dropout = DROPOUT
if hasattr(config, "attention_dropout"):
    config.attention_dropout = DROPOUT

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME_OR_PATH, config=config)
if hasattr(model, "roberta"):
    backbone = model.roberta
    layer_prefix = "roberta.encoder.layer."
    emb_module = backbone.embeddings
elif hasattr(model, "bert"):
    backbone = model.bert
    layer_prefix = "bert.encoder.layer."
    emb_module = backbone.embeddings
else:
    backbone = None
    layer_prefix = None
    emb_module = None

if emb_module is not None and FREEZE_EMBEDDINGS:
    for p in emb_module.parameters():
        p.requires_grad = False

if backbone is not None and hasattr(backbone, "encoder") and hasattr(backbone.encoder, "layer"):
    n_layers = len(backbone.encoder.layer)
    n_freeze = min(FREEZE_FIRST_N_LAYERS, n_layers)
    for i in range(n_freeze):
        for p in backbone.encoder.layer[i].parameters():
            p.requires_grad = False
    print(f"✅ Frozen embeddings={FREEZE_EMBEDDINGS} and first {n_freeze}/{n_layers} layers")
else:
    print("⚠️ Could not locate encoder layers; skipping freezing.")

no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]

def add_group(param_groups, named_params, lr):
    if not named_params:
        return
    decay_params, nodecay_params = [], []
    for n, p in named_params:
        if not p.requires_grad:
            continue
        if any(nd in n for nd in no_decay):
            nodecay_params.append(p)
        else:
            decay_params.append(p)
    if decay_params:
        param_groups.append({"params": decay_params, "lr": lr, "weight_decay": WEIGHT_DECAY})
    if nodecay_params:
        param_groups.append({"params": nodecay_params, "lr": lr, "weight_decay": 0.0})

def build_param_groups(model):
    groups = []
    head = [(n, p) for n, p in model.named_parameters() if "classifier" in n]
    add_group(groups, head, LR_HEAD)

    if layer_prefix is not None and backbone is not None and hasattr(backbone, "encoder"):
        n_layers = len(backbone.encoder.layer)
        cut = int(math.floor(n_layers * 2 / 3))
        early, late, other = [], [], []
        for n, p in model.named_parameters():
            if "classifier" in n:
                continue
            if n.startswith(layer_prefix):
                parts = n.split(".")
                idx = None
                try:
                    idx = int(parts[3])
                except Exception:
                    idx = None
                if idx is not None and idx < cut:
                    early.append((n, p))
                else:
                    late.append((n, p))
            else:
                other.append((n, p))
        add_group(groups, early, LR_EARLY)
        add_group(groups, late + other, LR_LATE)
    else:
        rest = [(n, p) for n, p in model.named_parameters() if "classifier" not in n]
        add_group(groups, rest, LR_LATE)

    return groups

class LayerwiseLRTrainer(Trainer):
    def create_optimizer(self):
        if self.optimizer is not None:
            return self.optimizer
        from torch.optim import AdamW
        self.optimizer = AdamW(build_param_groups(self.model))
        return self.optimizer


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Frozen embeddings=True and first 6/12 layers


In [ ]:
resume_ckpt = latest_checkpoint(RUN_DIR)
print("Resume checkpoint:", resume_ckpt)

args = TrainingArguments(
    output_dir=RUN_DIR,
    seed=SEED,

    per_device_train_batch_size=BSZ_TRAIN,
    per_device_eval_batch_size=BSZ_EVAL,
    gradient_accumulation_steps=GRAD_ACC,
    num_train_epochs=EPOCHS,
    max_grad_norm=MAX_GRAD_NORM,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    lr_scheduler_type="linear",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    label_smoothing_factor=LABEL_SMOOTH,
    logging_steps=50,
    report_to="none",
    fp16=False,
    bf16=True,
    dataloader_num_workers=2,
)

early_cb = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.0005
)

trainer = LayerwiseLRTrainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    data_collator=data_collator,
    train_dataset=train_tok,
    eval_dataset=dev_tok,
    compute_metrics=compute_metrics,
    callbacks=[early_cb],
)


Resume checkpoint: None


/tmp/ipython-input-2493019583.py:45: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `LayerwiseLRTrainer.__init__`. Use `processing_class` instead.
  trainer = LayerwiseLRTrainer(


In [ ]:
trainer.train(resume_from_checkpoint=resume_ckpt)

print("✅ Best checkpoint:", trainer.state.best_model_checkpoint)
print("✅ Best OOD-dev metrics:", trainer.evaluate(dev_tok))


Step,Training Loss,Validation Loss,F1 Macro
600,0.480400,0.361410,0.937600
1200,0.310500,0.220755,0.978945
1800,0.270800,0.212408,0.986902
2400,0.254800,0.189732,0.992019
3000,0.229500,0.191927,0.992004
3600,0.239000,0.187310,0.993158
4200,0.224400,0.186783,0.993729
4800,0.224200,0.189051,0.993154
5400,0.217800,0.185356,0.993156


✅ Best checkpoint: /content/drive/MyDrive/semeval_codebert_full500k_plus_bigaug_v3/checkpoint-3600


✅ Best OOD-dev metrics: {'eval_loss': 0.18731015920639038, 'eval_f1_macro': 0.993157670840402, 'eval_runtime': 1.4263, 'eval_samples_per_second': 1247.966, 'eval_steps_per_second': 19.631, 'epoch': 0.1728}


In [ ]:
val_tok = val_official.map(preprocess, batched=True)
val_tok = val_tok.remove_columns([c for c in val_tok.column_names if c not in keep_cols])

print("Official validation metrics:", trainer.evaluate(val_tok))


Official validation metrics: {'eval_loss': 0.2895880937576294, 'eval_f1_macro': 0.9401534849669039, 'eval_runtime': 66.7461, 'eval_samples_per_second': 1498.215, 'eval_steps_per_second': 23.417, 'epoch': 0.1728}


In [ ]:
test_tok = test_official.map(preprocess, batched=True)
keep_test_cols = ["input_ids", "attention_mask"]
if "label" in test_official.column_names:
    keep_test_cols.append("label")
test_tok = test_tok.remove_columns([c for c in test_tok.column_names if c not in keep_test_cols])

pred = trainer.predict(test_tok)
logits = pred.predictions
probs = softmax(logits, axis=-1)
y_hat = np.argmax(probs, axis=-1)

if "label" in test_official.column_names:
    y_true = np.array(test_official["label"])
    print("✅ TEST Macro F1:", f1_score(y_true, y_hat, average="macro"))
    print(classification_report(y_true, y_hat, target_names=["human(0)", "machine(1)"], digits=6))
    print("Confusion:\n", confusion_matrix(y_true, y_hat, labels=[0,1]))
else:
    print("ℹ️ Test has no labels. First 30 preds:", y_hat[:30])


✅ TEST Macro F1: 0.4029277542582652
              precision    recall  f1-score   support

    human(0)   0.924528  0.252252  0.396360       777
  machine(1)   0.262690  0.928251  0.409496       223

    accuracy                       0.403000      1000
   macro avg   0.593609  0.590252  0.402928      1000
weighted avg   0.776938  0.403000  0.399289      1000

Confusion:
 [[196 581]
 [ 16 207]]


In [ ]:
!ls /content | head


sample_data


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from google.colab import files
files.upload()


Saving test.parquet to test.parquet


In [ ]:
!mv /content/test.parquet "/content/drive/MyDrive/test.parquet"
!ls -lah "/content/drive/MyDrive/test.parquet"


-rw-------+ 1 root root 285M Jan 10 04:10 /content/drive/MyDrive/test.parquet


In [ ]:
!cp "/content/drive/MyDrive/test.parquet" /content/test.parquet
!ls -lah /content/test.parquet


-rw------- 1 root root 285M Jan 10 04:21 /content/test.parquet


In [ ]:
import pandas as pd
test_df = pd.read_parquet("/content/test.parquet", columns=["ID","code"])
print("rows:", len(test_df), "cols:", test_df.columns.tolist())
print("ID unique:", test_df["ID"].nunique())
test_df.head()


rows: 500000 cols: ['ID', 'code']
ID unique: 500000


,ID,code
0,0,"(H, W) = map(int, input().split())\nA = [list(..."
2,2,"n = int(input())\nt = list(map(int, input().sp..."
5,5,import sys\n\nclass LazySegmentTree:\n def ...
6,6,n = int(input())\ns = input()\ntup = []\nfor i...
7,7,import sys\ninput = sys.stdin.readline\nfrom m...


In [ ]:

!pip -q install -U transformers datasets accelerate safetensors scipy pyarrow
import os, glob, json
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification, Trainer, TrainingArguments
TEST_PATH = "/content/test.parquet"
CKPT_DIR = "/content/drive/MyDrive/semeval_codebert_full500k_plus_bigaug_v3/checkpoint-3600"

assert os.path.exists(TEST_PATH), f"❌ Nu gasesc: {TEST_PATH}"
assert os.path.exists(CKPT_DIR), f"❌ Nu gasesc CKPT_DIR: {CKPT_DIR}"
if not os.path.exists(os.path.join(CKPT_DIR, "config.json")):
    ts = os.path.join(CKPT_DIR, "trainer_state.json")
    picked = None
    if os.path.exists(ts):
        try:
            st = json.load(open(ts, "r", encoding="utf-8"))
            best = st.get("best_model_checkpoint")
            if best and os.path.exists(best):
                picked = best
        except Exception:
            pass
    if picked is None:
        cks = sorted(glob.glob(os.path.join(CKPT_DIR, "checkpoint-*")),
                     key=lambda p: int(p.split("-")[-1]) if p.split("-")[-1].isdigit() else -1)
        assert cks, f"❌ Nu gasesc checkpoint-* in {CKPT_DIR}"
        picked = cks[-1]
    CKPT_DIR = picked

print("✅ Using checkpoint:", CKPT_DIR)
tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR, use_fast=True)
config = AutoConfig.from_pretrained(CKPT_DIR)
model = AutoModelForSequenceClassification.from_pretrained(CKPT_DIR, config=config)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("✅ Device:", device)
test_df = pd.read_parquet(TEST_PATH, columns=["ID", "code"])
print("Test rows:", len(test_df), "| ID unique:", test_df["ID"].nunique())

hf_test = Dataset.from_pandas(test_df)
MAX_LEN = 384
def tok(batch):
    return tokenizer(batch["code"], truncation=True, max_length=MAX_LEN, padding=False)

test_tok = hf_test.map(tok, batched=True, batch_size=1000)
test_tok = test_tok.remove_columns([c for c in test_tok.column_names if c not in ["input_ids", "attention_mask"]])

args = TrainingArguments(
    output_dir="/content/tmp_pred",
    per_device_eval_batch_size=64,
    dataloader_num_workers=2,
    report_to="none",
)

pred_trainer = Trainer(model=model, args=args, tokenizer=tokenizer)
pred = pred_trainer.predict(test_tok)

y_hat = np.argmax(pred.predictions, axis=-1).astype(int)
assert len(y_hat) == len(test_df), "❌ mismatch preds vs test"
sub = pd.DataFrame({"ID": test_df["ID"].values, "label": y_hat})
out_path = "/content/submission.csv"
sub.to_csv(out_path, index=False)

print("✅ submission.csv ready:", out_path)
print(sub.head())
print("Rows:", len(sub))


✅ Using checkpoint: /content/drive/MyDrive/semeval_codebert_full500k_plus_bigaug_v3/checkpoint-3600
✅ Device: cuda
Test rows: 500000 | ID unique: 500000


Map:   0%|          | 0/500000 [00:00<?, ? examples/s]

/tmp/ipython-input-667749935.py:77: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  pred_trainer = Trainer(model=model, args=args, tokenizer=tokenizer)


✅ submission.csv ready: /content/submission.csv
   ID  label
0   0      0
1   2      0
2   5      1
3   6      0
4   7      0
Rows: 500000


In [ ]:
import os, pandas as pd

df = pd.read_csv("/content/submission.csv")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print("File size (MB):", round(os.path.getsize("/content/submission.csv")/(1024*1024), 2))
df.head()


Rows: 500000
Columns: ['ID', 'label']
File size (MB): 4.29


,ID,label
0,0,0
1,2,0
2,5,1
3,6,0
4,7,0


In [ ]:
from google.colab import files
files.download("/content/submission.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>